# Evaluación exploratoria congelada: straddle vs benchmark histórico

El protocolo fechado en `docs/straddle_terminal_move_exploratory_protocol_2026-09-27.md` se guarda antes de ejecutar esta notebook. No se prueban variantes, subperiodos, filtros o métricas adicionales.

## Regla operativa

Se usan solo filas con `straddle_mid_rel_t`, `historical_move_baseline_t` y `terminal_move_rel` simultáneamente disponibles. Desarrollo: T ≤ 2023-12-29. Evaluación histórica: t ≥ 2024-01-02 y resultado terminal conocido. Se ajustan dos regresiones lineales con intercepto por separado y se truncan ambas predicciones negativas a cero. La métrica principal es `MSE_baseline - MSE_straddle`; positiva favorece al straddle. El IC exploratorio usa bootstrap de bloques móviles de 30 fechas, 5,000 réplicas y semilla 20260927.

In [1]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display, Markdown

project_root = Path.cwd().resolve().parent
sys.path.insert(0, str(project_root / 'scripts'))
from evaluate_straddle_terminal_move_exploratory import run_frozen_evaluation, write_report
protocol_path = project_root / 'docs' / 'straddle_terminal_move_exploratory_protocol_2026-09-27.md'
assert protocol_path.exists(), 'El protocolo fechado debe existir antes de ejecutar.'
print('Protocolo congelado:', protocol_path)
print('Lógica reutilizable: scripts/evaluate_straddle_terminal_move_exploratory.py')

Protocolo congelado: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura\docs\straddle_terminal_move_exploratory_protocol_2026-09-27.md
Lógica reutilizable: scripts/evaluate_straddle_terminal_move_exploratory.py


In [2]:
result = run_frozen_evaluation()
report_path = write_report(result)
print('Reporte escrito:', report_path)
print(f"Filas comunes: {result['common_rows']}; desarrollo: {result['development_size']}; evaluación: {result['evaluation_size']}")
print('Desarrollo t:', result['development_dates'], 'T:', result['development_terminal_dates'])
print('Evaluación t:', result['evaluation_dates'], 'T:', result['evaluation_terminal_dates'])
assert (result['development']['expiration'] <= pd.Timestamp('2023-12-29')).all()
assert (result['evaluation']['observation_date'] >= pd.Timestamp('2024-01-02')).all()
print('Comprobación temporal del protocolo: correcta.')

Reporte escrito: C:\Users\avazq\OneDrive\Escritorio\cienciadatos\portafolio\quant-research-notes\opciones_notas\straddle_ATM_SPY_volatilidad_futura\reports\straddle_terminal_move_exploratory_evaluation.md
Filas comunes: 1382; desarrollo: 715; evaluación: 648
Desarrollo t: (Timestamp('2021-01-29 00:00:00'), Timestamp('2023-12-01 00:00:00')) T: (Timestamp('2021-03-01 00:00:00'), Timestamp('2023-12-29 00:00:00'))
Evaluación t: (Timestamp('2024-01-02 00:00:00'), Timestamp('2026-08-03 00:00:00')) T: (Timestamp('2024-02-02 00:00:00'), Timestamp('2026-08-31 00:00:00'))
Comprobación temporal del protocolo: correcta.


## Ajustes congelados en desarrollo

Las pendientes e interceptos se estiman una vez con resultados cuya T no rebasa el corte. No se vuelven a estimar con la evaluación histórica.

In [3]:
coefficients = pd.DataFrame([
    {'modelo': 'baseline histórico', **result['baseline_model']},
    {'modelo': 'straddle', **result['straddle_model']},
])
display(coefficients)
display(Markdown('Las predicciones de evaluación se truncaron a cero únicamente después de aplicar estos coeficientes congelados.'))

,modelo,intercept,slope
0,baseline histórico,0.044913,-0.152850
1,straddle,0.008171,0.755432


Las predicciones de evaluación se truncaron a cero únicamente después de aplicar estos coeficientes congelados.

## Métricas de evaluación histórica

MSE es la métrica primaria; RMSE y MAE se muestran solo para escala. El bootstrap remuestrea los mismos bloques de fechas para los dos errores, preservando el emparejamiento.

In [4]:
metrics = pd.DataFrame([
    {'predictor': 'baseline histórico', **result['baseline_metrics']},
    {'predictor': 'straddle', **result['straddle_metrics']},
])
display(metrics)
lower, upper = result['mse_difference_ci95']
summary = pd.DataFrame([{
    'MSE_baseline_minus_MSE_straddle': result['mse_difference_baseline_minus_straddle'],
    'bootstrap_ci95_lower': lower,
    'bootstrap_ci95_upper': upper,
    'decision_protocol': result['decision'],
}])
display(summary)
display(Markdown('**INCONCLUSA** si el IC contiene cero; de otro modo el signo del IC determina la etiqueta exploratoria. Esto no es una prueba prospectiva intacta ni evidencia de rentabilidad.'))

,predictor,mse,rmse,mae
0,baseline histórico,0.000719,0.026814,0.021481
1,straddle,0.000519,0.022777,0.018082


,MSE_baseline_minus_MSE_straddle,bootstrap_ci95_lower,bootstrap_ci95_upper,decision_protocol
0,0.0002,0.000119,0.000331,FAVORECE_STRADDLE


**INCONCLUSA** si el IC contiene cero; de otro modo el signo del IC determina la etiqueta exploratoria. Esto no es una prueba prospectiva intacta ni evidencia de rentabilidad.

## Conclusión de investigación (cierre)

La pregunta fue si el costo relativo de un straddle ATM de SPY anticipa el movimiento absoluto hasta su vencimiento mejor que el benchmark histórico especificado. El predictor straddle fue `(call_mid + put_mid) / SPY_close_t`; el benchmark fue el promedio de 252 ventanas `abs(SPY_close_u / SPY_close_(u-H) - K / SPY_close_t)` que terminan en `u ≤ t`; el resultado fue `abs(SPY_close_T - K) / SPY_close_t`.

Con coeficientes congelados en 715 observaciones de desarrollo con `T ≤ 2023-12-29`, la evaluación histórica común de 648 fechas de 2024–2026 obtuvo MSE 0.00071901 para el benchmark y 0.00051880 para el straddle: reducción de 27.8%. La diferencia `MSE_baseline - MSE_straddle` tuvo IC exploratorio 95% [0.00011880, 0.00033096]. Por ello, este resultado favorece al straddle frente a **este** benchmark en esta evaluación exploratoria.

La pendiente OLS negativa del benchmark en desarrollo (−0.15284988) fue auditada después: el linaje, H, k, 252 ventanas, fechas, límite `≤ t`, splits y promedios coincidieron en las 715 filas. No se encontró error de construcción; la pendiente describe una relación empírica de esa muestra, no una razón para cambiar el benchmark después de observarla.

## Alcance y límites

No se demostró superioridad frente a cualquier método histórico, rentabilidad de comprar opciones ni validez prospectiva. 2024–2026 ya fue visto y no es prueba prospectiva intacta. La disponibilidad EOD sigue siendo un supuesto no verificado; SPY tiene opciones americanas; y los bid/ask archivados no prueban ejecución simultánea. Por ello, prima y valor intrínseco no son P&L de una operación real.

## Mis notas

- 
- 
- 